*Don't forget to save a copy of this notebook in your Drive before working with it!*

# Folium and PyVis

## Folium

The Python package Folium is primarily used to create interactive geographic visualizations by acting as a bridge between Python data and the [Leaflet.js](https://leafletjs.com/) JavaScript library. It excels at transforming geospatial datasets into interactive web maps with minimal code—often as few as three lines.  So what does Folium bring to the table?

* Interactivity: Generated maps allow users to zoom, pan, and click on elements like markers, popups, and tooltips.
* Easy Sharing: Maps can be exported as standalone HTML files to be served on websites or embedded directly into [Jupyter Notebooks](https://jupyter.org/) for interactive exploration.
* Data Integration: It works seamlessly with Pandas DataFrames and GeoPandas, making it easy to bind data to geographical boundaries.
* Visualizations:
* Choropleth Maps: Shading regions (like countries or states) based on specific data values, such as population or disease spread.
   * Heatmaps: Visualizing the intensity of data points at specific locations.
   * Markers & Shapes: Adding customizable pins, circles, polygons, and rectangles to represent points of interest or defined areas.
* Customization: Supports numerous tile providers like OpenStreetMap, [Mapbox](https://www.mapbox.com/), and Stamen, allowing you to change the base map's aesthetic (e.g., watercolor, dark mode, or terrain).
* Rich Overlays: Supports adding multiple layers, including GeoJSON and TopoJSON, which can be toggled using a built-in layer control element.


In [ ]:
import folium

# Create a Folium map object, centered around a specific latitude and longitude
m = folium.Map(location=[45.5236, -122.6750], zoom_start=13)

# Add a marker to the map
folium.Marker(
    location=[45.5236, -122.6750],
    popup="Portland, Oregon",
    icon=folium.Icon(color="red")
).add_to(m)

# Display the map. In a Colab environment, this will render the map directly below the cell.
m

In [ ]:
import folium
import json
import requests

# URL for US states GeoJSON data (often used in Folium examples)
states_geojson_url = "https://raw.githubusercontent.com/python-visualization/folium-example-data/main/us_states.json"

# Fetch the GeoJSON data
# Colab environments usually have 'requests' installed, but you might need to install it with !pip install requests if not.
response = requests.get(states_geojson_url)
states_geojson = json.loads(response.text)

# Define states considered fully or partially west of the Mississippi River
# This is a simplified list for demonstration purposes.
states_west_of_mississippi = [
    "Arkansas", "Iowa", "Louisiana", "Minnesota", "Missouri",
    "North Dakota", "South Dakota", "Nebraska", "Kansas", "Oklahoma", "Texas",
    "New Mexico", "Colorado", "Wyoming", "Montana", "Idaho", "Utah", "Arizona",
    "Nevada", "California", "Oregon", "Washington"
]

# Create a Folium map object, centered roughly on the continental US
m_us = folium.Map(location=[39.8283, -98.5795], zoom_start=4)

# Define a style function to color states
def style_function(feature):
    state_name = feature['properties']['name']
    if state_name in states_west_of_mississippi:
        return {
            'fillColor': '#ffaf00', # Orange for states west of Mississippi
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }
    else:
        return {
            'fillColor': '#cccccc', # Light gray for other states
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }

# Add the GeoJSON to the map with the custom style
folium.GeoJson(
    states_geojson,
    name='US States',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['name'], aliases=['State'])
).add_to(m_us)

# Display the map
m_us

In [ ]:
import folium
import json

# We'll use the 'states_geojson' data that was loaded in the previous example.
# First, let's augment the GeoJSON data to include the name length for each state.
# This makes it easier to use in the style function and tooltip.

# Calculate min/max name lengths for the colormap
# This loop also adds the 'name_length' property to each feature
name_lengths = []
for feature in states_geojson['features']:
    state_name = feature['properties']['name']
    length = len(state_name)
    feature['properties']['name_length'] = length
    name_lengths.append(length)

min_len = min(name_lengths)
max_len = max(name_lengths)

# Create a Folium map object, centered roughly on the continental US
m_name_length = folium.Map(location=[39.8283, -98.5795], zoom_start=4)

# Define a colormap based on the range of name lengths
# We'll use a simple blue gradient, from light blue for shorter names to dark blue for longer names
colormap = folium.LinearColormap(
    colors=['#e6f7ff', '#003366'], # Light blue to Dark blue
    vmin=min_len,
    vmax=max_len,
    caption='Number of letters in State Name'
)

# Define a style function to color states based on their name length
def style_function_by_name_length(feature):
    name_length = feature['properties']['name_length']
    return {
        'fillColor': colormap(name_length), # Use the colormap to get the color
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.7
    }

# Add the GeoJSON to the map with the custom style
folium.GeoJson(
    states_geojson,
    name='US States by Name Length',
    style_function=style_function_by_name_length,
    tooltip=folium.GeoJsonTooltip(fields=['name', 'name_length'], aliases=['State', 'Name Length'])
).add_to(m_name_length)

# Add the colormap legend to the map
m_name_length.add_child(colormap)

# Display the map
m_name_length

## PyVis

PyVis is a Python library specifically designed for creating interactive network visualizations. Much like Folium acts as a bridge to Leaflet.js for maps, PyVis acts as a Python wrapper for the vis.js JavaScript library to handle complex, dynamic graphs.

What are the key features of PyVis?

* Dynamic Interactivity: Unlike static plotting libraries (like Matplotlib), PyVis creates graphs where you can drag and drop nodes, zoom, and hover to see tooltips.
* Physics Engine: It includes a built-in physics engine that allows nodes to bounce, repel, or attract each other, making the layout feel "alive" and helping to naturally organize complex clusters.
* HTML Output: It generates standalone HTML files containing all the necessary CSS and JavaScript. This makes it incredibly easy to embed your network graphs into websites or dashboards without needing a Python backend to view them.
* NetworkX Integration: It integrates seamlessly with [NetworkX](https://networkx.org/). You can perform complex graph theory calculations (like centrality) in NetworkX and then simply pass that object to PyVis for visualization.
* Configuration UI: One unique feature is the ability to add a live configuration menu directly onto the map, allowing users to tweak physics settings, colors, and layouts in real-time within the browser.

In [ ]:
!pip install pyvis
from pyvis.network import Network
import networkx as nx
from IPython.core.display import display, HTML

In [ ]:
G = nx.Graph()

# Add nodes with attributes (optional, but good for PyVis)
G.add_node(1, label='Node 1', color='blue', size=20)
G.add_node(2, label='Node 2', color='red', size=25)
G.add_node(3, label='Node 3', color='green', size=30)
G.add_node(4, label='Node 4', color='orange', size=20)
G.add_node(5, label='Node 5', color='purple', size=25)
G.add_node(6, label='Node 6', color='black', size=30)

# Add edges with attributes (optional)
G.add_edge(1, 2, weight=0.7, title='Connection A')
G.add_edge(1, 3, weight=0.3, title='Connection B')
G.add_edge(2, 4, weight=1.0, title='Connection C')
G.add_edge(3, 5, weight=0.5, title='Connection D')
G.add_edge(4, 6, weight=0.8, title='Connection E')
G.add_edge(5, 1, weight=0.6, title='Connection F')
G.add_edge(2, 5, weight=0.9, title='Connection G')
G.add_edge(3, 6, weight=0.4, title='Connection H')
G.add_edge(4, 5, weight=0.2, title='Connection I')

# 2. Create a PyVis network object
# 'notebook=True' allows it to render directly in Colab/Jupyter
net = Network(notebook=True, height='750px', width='100%',
              cdn_resources='remote', # Use remote CDN for resources
              directed=False)

# Optional: Configure physics and interaction settings using set_options
net.set_options("""
{
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -50,
            "centralGravity": 0.005,
            "springLength": 100,
            "springConstant": 0.18
        },
        "maxVelocity": 50,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {
            "enabled": true,
            "iterations": 2000,
            "updateInterval": 25
        }
    }
}
""")

# 3. Add the NetworkX graph to the PyVis network
net.from_nx(G)

# 4. Generate and display the interactive network visualization
# The network will be saved to an HTML file and displayed below the cell.
file_name = 'pyvis_example_network.html'
net.show(file_name)
display(HTML(file_name))

## A larger example with PyVis

The Cora data set: https://linqs.org/datasets/#cora

In [ ]:
import pandas as pd
original_links = pd.read_csv("https://github.com/cbrown-clu/class_data/raw/refs/heads/main/data/cora/cora.cites",
                             sep="\t",header=None)
original_links.columns = ["cited","citing"]
print(f"Data shape is {original_links.shape}")
original_links.head()

In [ ]:
id_values = pd.concat([original_links.cited,original_links.citing]).unique()
from sklearn.model_selection import train_test_split
train, test = train_test_split(original_links, test_size=0.7, random_state=42)
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Convert numpy.int64 IDs to standard Python int IDs before adding to NetworkX
G.add_nodes_from([int(x) for x in id_values])

# Add edges from the train set
# For each row, add an edge from 'citing' to 'cited', ensuring IDs are int
for index, row in train.iterrows():
    G.add_edge(int(row['citing']), int(row['cited']))
initial_nodes_count = G.number_of_nodes()

# Create an undirected version of the graph to find isolated nodes
G_undirected_temp = G.to_undirected()

# Find nodes with no edges (degree 0) in the undirected graph
isolated_nodes = [node for node, degree in G_undirected_temp.degree() if degree == 0]

# Remove these isolated nodes from the original directed graph G
G.remove_nodes_from(isolated_nodes)

In [ ]:
from pyvis.network import Network
from IPython.core.display import display, HTML

# Create a PyVis network object
# 'notebook=True' allows it to render directly in Colab/Jupyter
# Since G is a DiGraph, we set directed=True
net = Network(notebook=True, height='750px', width='100%',
              cdn_resources='remote',
              directed=True) # Set to True for a directed graph

# Optional: Configure physics and interaction settings using set_options
net.set_options("""
{
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -50,
            "centralGravity": 0.005,
            "springLength": 100,
            "springConstant": 0.18
        },
        "maxVelocity": 50,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {
            "enabled": true,
            "iterations": 2000,
            "updateInterval": 25
        }
    }
}
""")

# Add the NetworkX graph to the PyVis network
net.from_nx(G)

# Generate and display the interactive network visualization
file_name = 'pyvis_cora_network.html'
net.show(file_name)
display(HTML(file_name))